<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%A1%D0%B0%D0%B1_%D0%9F%D0%BE%D1%85_%D1%8E%D0%B7%D0%B5%D1%80%D1%8B_%D0%91%D0%B5%D0%B7_%D0%92%D0%B7%D0%B0%D0%B8%D0%BC_%D0%9E%D0%B3%D1%80_100_ml_ozon_recsys_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OZON RecSys Baseline - Рекомендательная система для категории Apparel
Этот ноутбук содержит базовое решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.

## Задача
- Предсказать топ-100 товаров для каждого пользователя из тестовой выборки
- Метрика оценки: NDCG@100
- Данные: ~38GB в формате parquet, 1.6B взаимодействий, 19M заказов


In [1]:
# === 1. Монтируем Google Drive, задаём пути к данным (структура Colab/Яндекс) ===
from google.colab import drive
drive.mount('/content/drive')

# Шаг 1. Импорты и пути (Colab/локально, без лишних библиотек)
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict, Counter, deque # Добавлены для подсчета популярности
import glob

# Пути к данным (замените на свои)
ORDERS_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_orders_data/final_apparel_orders_data_07'
TEST_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/ml_ozon_recsys_test'

Mounted at /content/drive


In [2]:
# Шаг 2. Загрузка заказов
def load_orders_single_path(path):
    print(f"Загружаем тренировочные данные заказов из: {path}")
    orders = []
    for f in Path(path).rglob('*.parquet'):
        orders.append(pd.read_parquet(f))
    df = pd.concat(orders, ignore_index=True)

    # Если created_date отсутствует, создаем её из created_timestamp
    if 'created_date' not in df.columns and 'created_timestamp' in df.columns:
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_timestamp'].dt.date
        df['created_date'] = pd.to_datetime(df['created_date'])

    print(f"Загружено заказов: {len(df):,}")
    return df

# Загружаем только из ORDERS_PATH
orders_df = load_orders_single_path(ORDERS_PATH)

Загружаем тренировочные данные заказов из: /content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_orders_data/final_apparel_orders_data_07
Загружено заказов: 1,475,216


In [3]:
print("=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(orders_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in orders_df.columns:
    print(f"  • {col}: {orders_df[col].dtype}")

=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|    |   item_id |   user_id | created_timestamp          | last_status      | last_status_timestamp   | created_date        |
+====+===========+===========+============================+==================+=========================+=====================+
|  0 | 332361399 |      2841 | 2025-07-14 08:38:22.250000 | proccesed_orders | 2025-07-14 11:35:22     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|  1 | 153141296 |      3681 | 2025-07-14 09:13:56.410000 | proccesed_orders | 2025-07-14 10:47:53     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+

📊 Схема данных:
  • ite

In [4]:
# Шаг 3. Загрузка тестовых пользователей
def load_test_users():
    print("Загружаем тестовых пользователей...")
    test_files = glob.glob(f'{TEST_PATH}/*.parquet')
    users = set()
    for f in tqdm(test_files, desc="Обработка тестовых файлов"):
        df_part = pd.read_parquet(f)
        if 'user_id' in df_part.columns:
            users.update(df_part['user_id'].unique())
    print(f"Найдено уникальных тестовых пользователей: {len(users):,}")
    return list(users)

test_users = load_test_users()

Загружаем тестовых пользователей...


Обработка тестовых файлов: 100%|██████████| 1/1 [00:01<00:00,  1.52s/it]

Найдено уникальных тестовых пользователей: 470,347


In [5]:
print("=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
test_users_df = pd.DataFrame({'user_id': test_users})
print(f"📊 Размер: {len(test_users_df):,} строк")
print("\n👉 Первые 2 строки:")
print(test_users_df.head(2).to_string())
print("\n📋 Колонки и типы данных:")
for col in test_users_df.columns:
    print(f"  • {col:20} {test_users_df[col].dtype}")

=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
📊 Размер: 470,347 строк

👉 Первые 2 строки:
   user_id
0        1
1  3145730

📋 Колонки и типы данных:
  • user_id              int32


In [6]:
print("\n\nАНАЛИЗ ЗАКАЗОВ")
print("=" * 50)
print(f"Общее количество заказов: {len(orders_df):,}")
print(f"Уникальных пользователей: {orders_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {orders_df['item_id'].nunique():,}")

if 'created_date' in orders_df.columns:
    min_date = orders_df['created_date'].min().date()
    max_date = orders_df['created_date'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'last_status' in orders_df.columns:
    print("\nРаспределение статусов заказов:")
    status_counts = orders_df['last_status'].value_counts()
    status_counts_pct = orders_df['last_status'].value_counts(normalize=True) * 100
    for status, count in status_counts.items():
        pct = status_counts_pct[status]
        print(f"  {status}: {count:,} ({pct:.1f}%)")




АНАЛИЗ ЗАКАЗОВ
Общее количество заказов: 1,475,216
Уникальных пользователей: 346,759
Уникальных товаров: 692,335
Период данных: 2025-07-02 - 2025-07-15

Распределение статусов заказов:
  proccesed_orders: 560,517 (38.0%)
  delivered_orders: 484,645 (32.9%)
  canceled_orders: 430,054 (29.2%)


In [7]:
print("\nТоп-10 самых популярных товаров (по количеству заказов):")
top_items_orders = orders_df['item_id'].value_counts().head(10)
for item_id, count in top_items_orders.items():
    print(f"  Товар {item_id}: {count:,} заказов")



Топ-10 самых популярных товаров (по количеству заказов):
  Товар 175287070: 768 заказов
  Товар 51974017: 757 заказов
  Товар 166327353: 606 заказов
  Товар 187052809: 544 заказов
  Товар 247423473: 433 заказов
  Товар 11083343: 424 заказов
  Товар 334086992: 408 заказов
  Товар 206494927: 402 заказов
  Товар 63987378: 384 заказов
  Товар 201930716: 342 заказов


In [8]:
# Шаг 5. Аналитическая "модель популярности" по тренировочным данным
def get_popular_items(orders_df, top_k=100):
    """Получить топ-K популярных товаров из тренировочной выборки"""
    print("\n=== 🏆 РАСЧЕТ ПОПУЛЯРНЫХ ТОВАРОВ (TRAIN) ===")
    item_counts = orders_df['item_id'].value_counts().head(top_k)
    popular_items = item_counts.index.tolist()

    print(f"Топ-{top_k} популярных товаров рассчитан")
    print("\n📋 Топ-10 самых популярных товаров:")
    for i, (item_id, count) in enumerate(item_counts.head(10).items(), 1):
        print(f"  {i:2d}. Товар {item_id:>12}: {count:>6,} заказов")

    return popular_items

popular_items = get_popular_items(orders_df, top_k=100)


=== 🏆 РАСЧЕТ ПОПУЛЯРНЫХ ТОВАРОВ (TRAIN) ===
Топ-100 популярных товаров рассчитан

📋 Топ-10 самых популярных товаров:
   1. Товар    175287070:    768 заказов
   2. Товар     51974017:    757 заказов
   3. Товар    166327353:    606 заказов
   4. Товар    187052809:    544 заказов
   5. Товар    247423473:    433 заказов
   6. Товар     11083343:    424 заказов
   7. Товар    334086992:    408 заказов
   8. Товар    206494927:    402 заказов
   9. Товар     63987378:    384 заказов
  10. Товар    201930716:    342 заказов


In [9]:
# Шаг 6. Получение тестовых пользователей
test_users_during_period = list(set(test_users))
print(f"\n=== 👥 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
print(f"Найдено тестовых пользователей: {len(test_users_during_period):,}")


=== 👥 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
Найдено тестовых пользователей: 470,347


In [10]:
# Шаг 7. Вычисление истории покупок для тестовых пользователей (их покупки во всем периоде)
def build_user_history_full_period(orders_df, test_users):
    """Построить историю покупок тестовых пользователей во всем периоде"""
    print("\n=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (весь период) ===")

    # Все доставленные заказы тестовых пользователей во всем периоде
    user_history = orders_df[
        (orders_df['last_status'] == 'delivered_orders') &
        (orders_df['user_id'].isin(test_users))
    ].groupby('user_id')['item_id'].apply(set).to_dict()

    # Конвертируем в список для совместимости
    user_preferences = {uid: list(items) for uid, items in user_history.items()}

    print(f"История покупок построена для {len(user_preferences):,} пользователей")
    return user_preferences

user_preferences = build_user_history_full_period(orders_df, test_users_during_period)


=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (весь период) ===
История покупок построена для 211,531 пользователей


In [11]:
# Шаг 8. Генерация рекомендаций
def generate_recommendations(test_users, popular_items, user_preferences, top_k=100):
    """Генерация рекомендаций: популярные товары, исключая уже купленные"""
    print("\n=== 🎯 ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ ===")
    recs = {}
    for uid in tqdm(test_users, desc='Генерация рекомендаций'):
        bought = set(user_preferences.get(uid, []))
        recs[uid] = [item for item in popular_items if item not in bought][:top_k]
    print(f"Рекомендации сгенерированы для {len(recs):,} пользователей")
    return recs

recommendations = generate_recommendations(test_users_during_period, popular_items, user_preferences)


=== 🎯 ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ ===


Генерация рекомендаций: 100%|██████████| 470347/470347 [00:03<00:00, 139385.20it/s]

Рекомендации сгенерированы для 470,347 пользователей


In [13]:
from scipy.sparse import csr_matrix
import numpy as np
from tqdm import tqdm
import pickle
from pathlib import Path

# === 1. Создание матрицы user-item (оптимизированная версия) ===
print("Создаем оптимизированную user-item матрицу...")

# Получаем всех уникальных пользователей и товаров
train_users = orders_df['user_id'].unique()
train_items = orders_df['item_id'].unique()

# Создаем маппинги
user_to_idx = {user: idx for idx, user in enumerate(train_users)}
idx_to_user = {idx: user for user, idx in user_to_idx.items()}
item_to_idx = {item: idx for idx, item in enumerate(train_items)}

# Создаем разреженную матрицу (только доставленные заказы)
delivered_orders = orders_df[orders_df['last_status'] == 'delivered_orders']

rows, cols, data = [], [], []

for _, row in tqdm(delivered_orders.iterrows(), total=len(delivered_orders), desc="Построение матрицы"):
    if row['user_id'] in user_to_idx and row['item_id'] in item_to_idx:
        user_idx = user_to_idx[row['user_id']]
        item_idx = item_to_idx[row['item_id']]
        rows.append(user_idx)
        cols.append(item_idx)
        data.append(1)  # Бинарная метка

# Создаем CSR матрицу
user_item_matrix = csr_matrix((data, (rows, cols)),
                             shape=(len(train_users), len(train_items)))

print(f"Матрица создана: {user_item_matrix.shape[0]} users × {user_item_matrix.shape[1]} items")
print(f"Плотность: {(user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]) * 100):.6f}%")

Создаем оптимизированную user-item матрицу...


Построение матрицы: 100%|██████████| 484645/484645 [00:32<00:00, 15062.40it/s]


Матрица создана: 346759 users × 692335 items
Плотность: 0.000201%


In [30]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# === 3. Создание словаря похожих пользователей (на основе матрицы взаимодействий) ===
print("\n=== 🚀 СОЗДАНИЕ СЛОВАРЯ ПОХОЖИХ ПОЛЬЗОВАТЕЛЕЙ (МАТРИЦА ВЗАИМОДЕЙСТВИЙ) ===")

similar_users_dict = {}
test_users_in_train = [uid for uid in test_users_during_period if uid in user_to_idx]
print(f"Найдено {len(test_users_in_train)} тестовых пользователей в тренировочных данных")

# Создаем матрицу взаимодействий пользователь-товар на основе доставленных заказов
print("Создаем матрицу взаимодействий...")
delivered_orders = orders_df[orders_df['last_status'] == 'delivered_orders']

# Используем уже созданную матрицу user_item_matrix
print(f"Используем существующую матрицу: {user_item_matrix.shape}")

# Параметры
NUM_CANDIDATES = 5  # Целевое количество похожих пользователей
BATCH_SIZE = 10000   # Размер батча для обработки

# Обрабатываем пользователей батчами
num_batches = (len(test_users_in_train) + BATCH_SIZE - 1) // BATCH_SIZE
users_with_similar = 0

for batch_idx in range(num_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(test_users_in_train))
    batch_users = test_users_in_train[start_idx:end_idx]

    print(f"Обработка батча {batch_idx+1}/{num_batches} ({len(batch_users)} пользователей)")
    batch_indices = [user_to_idx[uid] for uid in batch_users]

    # Вычисляем косинусное сходство между пользователями в батче и всеми пользователями
    batch_vectors = user_item_matrix[batch_indices]

    # Вычисляем сходство для всего батча сразу
    similarities = cosine_similarity(batch_vectors, user_item_matrix)

    batch_with_similar = 0

    for i, test_user_id in enumerate(batch_users):
        # Получаем сходство текущего пользователя со всеми остальными
        user_similarities = similarities[i]

        # Находим индексы наиболее похожих пользователей (кроме самого себя)
        test_user_idx = user_to_idx[test_user_id]
        top_indices = np.argsort(user_similarities)[::-1][1:NUM_CANDIDATES*3+1]  # Исключаем самого себя

        similar_users = []
        for idx in top_indices:
            if idx != test_user_idx and idx in idx_to_user:
                similarity = user_similarities[idx]
                if similarity > 0:  # Отбрасываем отрицательные сходства
                    similar_user_id = idx_to_user[idx]
                    if similar_user_id != test_user_id:  # Дополнительная проверка
                        similar_users.append((similar_user_id, float(similarity)))

                        if len(similar_users) >= NUM_CANDIDATES:
                            break

        similar_users_dict[test_user_id] = similar_users

        if similar_users:
            users_with_similar += 1
            batch_with_similar += 1

    # Выводим промежуточную статистику
    processed = min((batch_idx + 1) * BATCH_SIZE, len(test_users_in_train))
    print(f"Прогресс: {processed}/{len(test_users_in_train)} ({processed/len(test_users_in_train)*100:.1f}%)")
    print(f"В текущем батче пользователей с похожими: {batch_with_similar}/{len(batch_users)} ({batch_with_similar/len(batch_users)*100:.1f}%)")
    print(f"Всего пользователей с похожими: {users_with_similar}/{processed} ({users_with_similar/processed*100:.1f}%)")

print(f"✅ Словарь похожих пользователей создан для {len(similar_users_dict)} пользователей")
print(f"🎯 Пользователей с хотя бы одним похожим: {users_with_similar} ({users_with_similar/len(similar_users_dict)*100:.1f}%)")



=== 🚀 СОЗДАНИЕ СЛОВАРЯ ПОХОЖИХ ПОЛЬЗОВАТЕЛЕЙ (МАТРИЦА ВЗАИМОДЕЙСТВИЙ) ===
Найдено 309565 тестовых пользователей в тренировочных данных
Создаем матрицу взаимодействий...
Используем существующую матрицу: (346759, 692335)
Обработка батча 1/31 (10000 пользователей)
Прогресс: 10000/309565 (3.2%)
В текущем батче пользователей с похожими: 4948/10000 (49.5%)
Всего пользователей с похожими: 4948/10000 (49.5%)
Обработка батча 2/31 (10000 пользователей)
Прогресс: 20000/309565 (6.5%)
В текущем батче пользователей с похожими: 4934/10000 (49.3%)
Всего пользователей с похожими: 9882/20000 (49.4%)
Обработка батча 3/31 (10000 пользователей)
Прогресс: 30000/309565 (9.7%)
В текущем батче пользователей с похожими: 5058/10000 (50.6%)
Всего пользователей с похожими: 14940/30000 (49.8%)
Обработка батча 4/31 (10000 пользователей)
Прогресс: 40000/309565 (12.9%)
В текущем батче пользователей с похожими: 4949/10000 (49.5%)
Всего пользователей с похожими: 19889/40000 (49.7%)
Обработка батча 5/31 (10000 пользоват

In [31]:
# === Статистика по словарю похожих пользователей ===
print("\n📊 СТАТИСТИКА СЛОВАРЯ ПОХОЖИХ ПОЛЬЗОВАТЕЛЕЙ")
print("=" * 50)

# Базовая статистика
similarity_counts = [len(similar) for similar in similar_users_dict.values()]
print(f"Всего пользователей в словаре: {len(similar_users_dict):,}")

print(f"\n📈 Распределение по количеству похожих:")
print(f"  Среднее количество похожих пользователей: {np.mean(similarity_counts):.2f}")
print(f"  Медиана: {np.median(similarity_counts):.1f}")
print(f"  Максимальное количество: {max(similarity_counts)}")
print(f"  Минимальное количество: {min(similarity_counts)}")

# Подсчет пользователей по категориям
zero_similar = sum(1 for x in similarity_counts if x == 0)
one_similar = sum(1 for x in similarity_counts if x == 1)
two_similar = sum(1 for x in similarity_counts if x == 2)
three_similar = sum(1 for x in similarity_counts if x == 3)
four_similar = sum(1 for x in similarity_counts if x == 4)

print(f"\n📋 Детальное распределение:")
print(f"  Без похожих: {zero_similar:,} ({zero_similar/len(similarity_counts)*100:.1f}%)")
print(f"  1 похожий:   {one_similar:,} ({one_similar/len(similarity_counts)*100:.1f}%)")
print(f"  2 похожих:   {two_similar:,} ({two_similar/len(similarity_counts)*100:.1f}%)")
print(f"  3 похожих:   {three_similar:,} ({three_similar/len(similarity_counts)*100:.1f}%)")
print(f"  4 похожих:   {four_similar:,} ({four_similar/len(similarity_counts)*100:.1f}%)")

# Статистика по схожести
all_similarities = []
for user_similar in similar_users_dict.values():
    for _, similarity in user_similar:
        all_similarities.append(similarity)

if all_similarities:
    print(f"\n📊 Статистика значений схожести:")
    print(f"  Средняя схожесть: {np.mean(all_similarities):.4f}")
    print(f"  Медианная схожесть: {np.median(all_similarities):.4f}")
    print(f"  Минимальная схожесть: {np.min(all_similarities):.4f}")
    print(f"  Максимальная схожесть: {np.max(all_similarities):.4f}")

# Пример структуры данных
print(f"\n🔍 Пример структуры словаря:")
sample_user_id = list(similar_users_dict.keys())[0] if similar_users_dict else None
if sample_user_id:
    sample_data = similar_users_dict[sample_user_id]
    print(f"  Пользователь {sample_user_id}:")
    print(f"    Тип: {type(sample_data)}")
    print(f"    Количество: {len(sample_data)}")
    if sample_data:
        print(f"    Первый элемент: {sample_data[0]}")
        print(f"    Тип элемента: {type(sample_data[0])}")

print("\n" + "=" * 50)


📊 СТАТИСТИКА СЛОВАРЯ ПОХОЖИХ ПОЛЬЗОВАТЕЛЕЙ
Всего пользователей в словаре: 309,565

📈 Распределение по количеству похожих:
  Среднее количество похожих пользователей: 1.82
  Медиана: 0.0
  Максимальное количество: 5
  Минимальное количество: 0

📋 Детальное распределение:
  Без похожих: 156,054 (50.4%)
  1 похожий:   28,577 (9.2%)
  2 похожих:   18,061 (5.8%)
  3 похожих:   12,991 (4.2%)
  4 похожих:   9,728 (3.1%)

📊 Статистика значений схожести:
  Средняя схожесть: 0.5630
  Медианная схожесть: 0.5000
  Минимальная схожесть: 0.0295
  Максимальная схожесть: 1.0000

🔍 Пример структуры словаря:
  Пользователь 1:
    Тип: <class 'list'>
    Количество: 0



In [32]:
# === 5. Сохранение данных ===
# Создаем путь для сохранения
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/processed'
Path(SAVE_PATH).mkdir(parents=True, exist_ok=True)

print("=== 💾 СОХРАНЕНИЕ ДАННЫХ ДЛЯ БЫСТРОЙ ЗАГРУЗКИ ===")

# Сохраняем словарь похожих пользователей
similar_users_file = f"{SAVE_PATH}/similar_users_dict.pkl"
with open(similar_users_file, 'wb') as f:
    pickle.dump(similar_users_dict, f)
print(f"✅ Похожие пользователи сохранены: {similar_users_file}")

=== 💾 СОХРАНЕНИЕ ДАННЫХ ДЛЯ БЫСТРОЙ ЗАГРУЗКИ ===
✅ Похожие пользователи сохранены: /content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/processed/similar_users_dict.pkl


In [ ]:
import numpy as np
from collections import defaultdict, Counter
from tqdm import tqdm

def generate_recommendations_hybrid(
    test_users,
    popular_items,
    user_preferences,
    similar_users_dict,
    orders_df, # Нам понадобится полный датафрейм для получения покупок похожих пользователей
    top_k=100
):
    """
    Генерация гибридных рекомендаций:
    - Для пользователей с историей и похожими: CF на базе пользователей.
    - Для остальных: популярные товары, исключая уже купленные.
    """
    print("\n=== 🎯 ГЕНЕРАЦИЯ ГИБРИДНЫХ РЕКОМЕНДАЦИЙ ===")

    # 1. Подготовка: создадим словарь покупок для быстрого доступа
    # Это нужно, чтобы быстро получить список покупок для любого пользователя
    print("Подготовка индекса покупок пользователей...")
    user_items_index = orders_df[
        (orders_df['last_status'] == 'delivered_orders')
    ].groupby('user_id')['item_id'].apply(set).to_dict()
    print(f"Индекс покупок создан для {len(user_items_index)} пользователей.")

    recommendations = {}

    for uid in tqdm(test_users, desc='Генерация рекомендаций'):
        rec_items = []

        # Получаем историю текущего пользователя (то, что он уже купил)
        bought_by_user = set(user_preferences.get(uid, []))

        # Получаем список похожих пользователей
        similar_users_list = similar_users_dict.get(uid, [])

        # --- Логика гибридной модели ---
        if uid in user_preferences and similar_users_list:
            # --- Сценарий 1: Есть история и есть похожие пользователи ---
            # Используем Collaborative Filtering

            # Счетчик для товаров, купленных похожими пользователями
            # Ключ: item_id, Значение: сумма "весов" (схожестей) пользователей, купивших его
            item_scores = defaultdict(float)

            # Проходим по каждому похожему пользователю
            for similar_user_id, similarity_score in similar_users_list:
                # Получаем товары, купленные похожим пользователем
                items_bought_by_similar = user_items_index.get(similar_user_id, set())

                # Добавляем эти товары в счетчик, взвешивая по степени схожести
                for item in items_bought_by_similar:
                    # Убедимся, что товар еще не куплен целевым пользователем
                    if item not in bought_by_user:
                         # Увеличиваем "счет" товара на "силу" схожести похожего пользователя
                        item_scores[item] += similarity_score

            # Сортируем товары по убыванию "счета" (релевантности)
            # item_scores.items() дает пары (item_id, score)
            sorted_items = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)

            # Извлекаем только ID товаров, отсекаем по top_k
            rec_items = [item_id for item_id, score in sorted_items[:top_k]]

            # Дополнительная проверка: если рекомендаций меньше top_k,
            # дополним популярными товарами (исключая уже купленные и уже добавленные)
            if len(rec_items) < top_k:
                # Множество уже рекомендованных товаров, чтобы не дублировать
                rec_items_set = set(rec_items)
                # Добавляем популярные товары, пока не наберем top_k
                for pop_item in popular_items:
                     if len(rec_items) >= top_k:
                        break
                     if pop_item not in bought_by_user and pop_item not in rec_items_set:
                         rec_items.append(pop_item)
                         rec_items_set.add(pop_item) # Обновляем множество для следующих итераций

        else:
            # --- Сценарий 2: Нет истории или нет похожих ---
            # Используем базовый подход: популярные, исключая купленные
            rec_items = [item for item in popular_items if item not in bought_by_user][:top_k]

        # Сохраняем рекомендации для пользователя
        recommendations[uid] = rec_items

    print(f"Рекомендации сгенерированы для {len(recommendations):,} пользователей")
    return recommendations

# --- Вызов функции ---
# Предполагается, что все необходимые переменные уже определены:
# popular_items, user_preferences, similar_users_dict, orders_df, test_users_during_period

# recommendations_hybrid = generate_recommendations_hybrid(
#     test_users=test_users_during_period,
#     popular_items=popular_items,
#     user_preferences=user_preferences,
#     similar_users_dict=similar_users_dict,
#     orders_df=orders_df, # Передаем весь датафрейм для построения индекса
#     top_k=100
# )



In [ ]:
recommendations_hybrid = generate_recommendations_hybrid(
    test_users=test_users_during_period,
    popular_items=popular_items,
    user_preferences=user_preferences,
    similar_users_dict=similar_users_dict,
    orders_df=orders_df, # Передаем весь датафрейм для построения индекса
    top_k=100
)

In [ ]:
# Шаг 7. Формирование submission-файла (готово к сабмиту)
def save_submission(recommendations_hybrid, filename='submission.csv'):
    rows = []
    for uid, items in tqdm(recommendations_hybrid.items(), desc='Формирование submission'):
        rows.append({
            'user_id': uid,
            'item_id_1 item_id_2 ... item_id_100': ' '.join(map(str, items))
        })
    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    print(f"Готово: {filename}")

save_submission(recommendations_hybrid, filename='ozon_baseline_analytic_submission.csv')